# HEFT operators with pmwd

Compute five hybrid effective field theory (HEFT) operators from white noise using the cosmology and settings of the [pmwd quickstart](quickstart.ipynb).

The first example calls `simulate_heft` in memory. The second saves and reloads a pmwd snapshot, then computes the operators from its saved inputs.

In [ ]:
import os

# Optional: select a GPU before importing JAX.
# os.environ['CUDA_VISIBLE_DEVICES'] = '0'
# Uncomment to select CPU execution.
# os.environ['JAX_PLATFORMS'] = 'cpu'

# GPU memory preallocation fraction, as in the quickstart.
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '.95'

import jax
import jax.numpy as jnp

print('JAX backend:', jax.default_backend())
print('JAX devices:', jax.devices())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from pmwd import Configuration, Cosmology, white_noise
from pmwd.heft import simulate_heft

## Configuration and initial conditions

The periodic box has side $256\,h^{-1}\mathrm{Mpc}$. As in the quickstart, the particle grid is $256^3$ on GPU (spacing $1\,h^{-1}\mathrm{Mpc}$) or $64^3$ on CPU (spacing $4\,h^{-1}\mathrm{Mpc}$). The force mesh has twice as many cells per axis; the HEFT output uses the particle-grid resolution.

The defaults use double precision for cosmological parameters and single precision for particle displacements and painted fields.

In [ ]:
if jax.default_backend() == 'gpu':
    ptcl_spacing = 1.  # Particle spacing in Mpc/h.
    ptcl_grid_shape = (256,) * 3
else:
    ptcl_spacing = 4.
    ptcl_grid_shape = (64,) * 3

conf = Configuration(ptcl_spacing, ptcl_grid_shape, mesh_shape=2)
output_grid = conf.ptcl_grid_shape[0]
print(f'{conf.ptcl_num:,} particles, force mesh {conf.mesh_shape}, '
      f'{conf.a_nbody_num} N-body steps, HEFT grid {(output_grid,) * 3}')

The default pmwd calculation uses the Eisenstein–Hu transfer fit, integrated growth, 2LPT, and N-body evolution.

In [ ]:
cosmo = Cosmology(conf, A_s_1e9=2.0, n_s=0.96, Omega_m=0.3,
                  Omega_b=0.05, h=0.7)
seed = 0
modes = white_noise(seed, conf)
a_ref = 1.

## Construct all five fields

The same unsmoothed linear modes feed the particle dynamics and the HEFT calculation. For the HEFT weights, this example smooths the Lagrangian linear density with the field-amplitude window

$$
d(\mathbf{k}) = W_{\mathrm G}(k)\,\delta_{\mathrm L}(\mathbf{k}, a_{\mathrm{ref}}),
\qquad
W_{\mathrm G}(k) = \exp\!\left[-\frac{k^2}{2k_d^2}\right],
\qquad
k_d = 0.75\,h\,\mathrm{Mpc}^{-1}.
$$

The window is applied before taking derivatives or forming quadratic products. It affects the four density-dependent weights; it does not change the particle trajectories or the matter overdensity `O1`.

The rows below follow the returned channel order.

| Output | Lagrangian particle weight | Affected by this smoothing window? |
| :--- | :--- | :--- |
| `O1`: matter overdensity $\delta_m$ | $1$ for every particle | No |
| `Od`: advected linear density | $d$ | Yes |
| `OL`: advected Laplacian | $\nabla^2 d$ | Yes |
| `Od2`: advected density squared | $\mathrm{d2}=d^2-\langle d^2\rangle_q$ | Yes |
| `Os2`: advected tidal squared | $\mathrm{s2}=\sum_{ij}K_{ij}^2-\langle\sum_{ij}K_{ij}^2\rangle_q$ | Yes |

Here $K_{ij}(\mathbf k)=(k_i k_j/k^2-\delta_{ij}/3)\,d(\mathbf k)$ is the traceless tidal tensor for $k\ne0$, with $K_{ij}(\mathbf0)=0$.

`smoothing='csst'` uses the window from [Zhou et al. (2025), arXiv:2506.04671](https://arxiv.org/abs/2506.04671) ([CSSTLaB](https://github.com/ShurenZhou1999/csstlab)):

$$
W_{\mathrm{CSST}}(k) =
\exp\!\left[-\frac{R^2}{4}\bigl(\max(k-k_{\mathrm{pivot}},0)\bigr)^2\right],
\qquad
R = 5\,h^{-1}\mathrm{Mpc},
\qquad
k_{\mathrm{pivot}} = 1\,h\,\mathrm{Mpc}^{-1}.
$$

Here $R$ and $k_{\mathrm{pivot}}$ correspond to `radius` and `pivot`. The field-amplitude window is the square root of the paper's power-spectrum window. `smoothing='none'` applies no additional window.

`a_ref=1` sets the linear-weight reference epoch to $z=0$; `conf.a_stop=1` sets the evolved-particle epoch to $z=0$.

[`linear_field_from_modes`](../../pmwd/heft.py) requires $0<a_{\mathrm{ref}}\leq a_{\mathrm{stop}}$ to stay within the computed growth table; `simulate_heft` enforces the same bounds.

In [ ]:
operators = simulate_heft(
    modes, cosmo, conf, a_ref=a_ref, output_grid=output_grid,
    smoothing='gaussian', kd=0.75,
)
operators = jax.block_until_ready(operators)

In [ ]:
labels = ['O1: matter overdensity', 'Od: linear density', 'OL: Laplacian',
          'Od2: centered density squared', 'Os2: centered tidal squared']
fields = np.asarray(operators)
assert fields.shape == (5, output_grid, output_grid, output_grid)
assert fields.dtype == conf.float_dtype
print('shape:', fields.shape, 'dtype:', fields.dtype)
for label, field in zip(labels, fields):
    assert np.isfinite(field).all()
    mean = field.mean(dtype=np.float64)
    rms = np.sqrt(np.mean(field**2, dtype=np.float64))
    if field.dtype == np.float64:
        assert abs(mean) <= 2e-11
    print(f'{label:29s} mean={mean: .3e}  rms={rms:.3e}')

### Visualize the fields

Each panel shows the central $z$ slice. Its symmetric color limits use the 99th percentile of the absolute slice values; values beyond those limits saturate at the end colors.

In [ ]:
def plot_operators(fields, box_size, title):
    fig, axes = plt.subplots(1, 5, figsize=(17, 3.8), layout='constrained')
    for ax, field, label in zip(axes, fields, labels):
        section = field[:, :, field.shape[2] // 2].T
        limit = np.percentile(np.abs(section), 99)
        limit = limit if limit > 0 else 1.
        picture = ax.imshow(
            section, origin='lower', extent=(0, box_size, 0, box_size),
            cmap='RdBu_r', vmin=-limit, vmax=limit,
        )
        ax.set_title(label, fontsize=10)
        ax.set_xlabel(r'$x\ [h^{-1}\mathrm{Mpc}]$')
        fig.colorbar(picture, ax=ax, shrink=0.7, extend='both')
    axes[0].set_ylabel(r'$y\ [h^{-1}\mathrm{Mpc}]$')
    fig.suptitle(title)
    plt.show()


plot_operators(fields, conf.box_size[0], 'End-to-end HEFT fields')

## Save the final fields (optional)

Set `save_final=True` to write the five fields to `heft_operators.npz`. To recompute them with another smoothing window, save the simulation inputs as shown in the next section.

In [ ]:
save_final = False
if save_final:
    from pathlib import Path
    destination = Path('heft_operators.npz')
    with destination.open('xb') as stream:
        np.savez_compressed(
            stream, operators=fields, labels=np.asarray(labels),
            a_ref=a_ref, a_stop=conf.a_stop, box_size=conf.box_size[0],
            smoothing='gaussian', kd=0.75,
        )
    print(f'Saved {destination}')

## Construct HEFT fields from saved pmwd outputs

`heft_operators` needs the unsmoothed linear density and evolved particles, retaining their original C-order Lagrangian association (last axis fastest). The final matter-density mesh alone does not contain this information.

The cell below generates a snapshot with native pmwd steps, repeating the evolution once for this demonstration. If `snapshot_path` already exists, it reuses the file. Choose a new path when changing the simulation settings.

| Entry | Meaning |
| :--- | :--- |
| `linear_field` | Unsmoothed linear density at `a_ref`, as a complex rFFT with `norm='forward'` |
| `pmid`, `disp` | Particle coordinates, $\mathbf x=\mathbf{pmid}\,\Delta x_{\mathrm{force}}+\mathbf{disp}$ |
| `configuration` | Geometry, units, precision, and simulation settings |
| `a_ref`, `output_grid`, `fft_norm` | Reference epoch and output conventions |
| `cosmology`, `seed` | Parameters used to generate the snapshot |

### Save a pmwd snapshot

`linear_field_from_modes` converts the native modes to the HEFT convention at `a_ref`, before smoothing. For an existing simulation, use its native linear modes and evolved `particles` in the save block.

In [ ]:
import json
from dataclasses import asdict
from pathlib import Path

from pmwd import Particles, boltzmann, linear_modes, lpt, nbody
from pmwd.heft import heft_operators, linear_field_from_modes

snapshot_path = Path('heft_example_data/pmwd_snapshot.npz')

if not snapshot_path.exists():
    snapshot_cosmo = boltzmann(cosmo, conf, varlin=False)
    native_modes = linear_modes(modes, snapshot_cosmo, conf)
    particles, observables = lpt(native_modes, snapshot_cosmo, conf)
    particles, _ = nbody(particles, observables, snapshot_cosmo, conf)
    linear_field = linear_field_from_modes(
        native_modes, snapshot_cosmo, conf, a_ref=a_ref,
    ).astype(jnp.promote_types(conf.float_dtype, jnp.complex64))
    jax.block_until_ready((linear_field, particles))

    configuration = asdict(conf)
    for name in ('cosmo_dtype', 'pmid_dtype', 'float_dtype'):
        configuration[name] = str(configuration[name])
    configuration = json.dumps(
        configuration, default=lambda value: np.asarray(value).tolist(),
    )
    cosmology = {
        name: float(getattr(cosmo, name))
        for name in ('A_s_1e9', 'n_s', 'Omega_m', 'Omega_b', 'h')
    }
    snapshot_path.parent.mkdir(parents=True, exist_ok=True)
    with snapshot_path.open('xb') as stream:
        np.savez(
            stream, linear_field=np.asarray(linear_field),
            pmid=np.asarray(particles.pmid), disp=np.asarray(particles.disp),
            configuration=configuration, cosmology=json.dumps(cosmology),
            seed=seed, a_ref=a_ref, output_grid=output_grid, fft_norm='forward',
        )
    print(f'Saved {snapshot_path}')
    del particles, observables, native_modes, linear_field, snapshot_cosmo
else:
    print(f'Reusing {snapshot_path}')

### Reload the snapshot

Read the saved arrays and configuration without recomputing linear theory or particle evolution. The linear field already includes growth at `a_ref`.

In [ ]:
with np.load(snapshot_path, allow_pickle=False) as snapshot:
    configuration = json.loads(str(snapshot['configuration']))
    for name in ('ptcl_grid_shape', 'mesh_shape', 'growth_inistep'):
        if isinstance(configuration[name], list):
            configuration[name] = tuple(configuration[name])
    configuration['symp_splits'] = tuple(
        tuple(pair) for pair in configuration['symp_splits']
    )
    loaded_conf = Configuration(**configuration)
    loaded_linear_field = jnp.asarray(snapshot['linear_field'])
    loaded_particles = Particles(
        loaded_conf, pmid=snapshot['pmid'], disp=snapshot['disp'],
    )
    loaded_a_ref = float(snapshot['a_ref'])
    loaded_output_grid = int(snapshot['output_grid'])
    assert str(snapshot['fft_norm']) == 'forward'

print(f'Loaded {loaded_conf.ptcl_num:,} particles; '
      f'a_ref={loaded_a_ref:g}, a_stop={loaded_conf.a_stop:g}; '
      f'HEFT grid {(loaded_output_grid,) * 3}')

### Compute and plot the five fields

The Gaussian window reproduces the first example when the snapshot uses the same simulation settings. Set `loaded_smoothing='csst'` or `'none'` to change the weights while keeping the particle trajectories fixed.

In [ ]:
loaded_smoothing = 'gaussian'
loaded_operators = heft_operators(
    loaded_linear_field, loaded_particles, loaded_conf,
    output_grid=loaded_output_grid, smoothing=loaded_smoothing,
    kd=0.75, radius=5., pivot=1.,
)
loaded_operators = jax.block_until_ready(loaded_operators)
loaded_fields = np.asarray(loaded_operators)
assert loaded_fields.shape == (5, loaded_output_grid, loaded_output_grid, loaded_output_grid)
assert loaded_fields.dtype == loaded_conf.float_dtype
assert np.isfinite(loaded_fields).all()
print('shape:', loaded_fields.shape, 'dtype:', loaded_fields.dtype)

In [ ]:
plot_operators(
    loaded_fields, loaded_conf.box_size[0],
    f'HEFT fields from saved pmwd outputs ({loaded_smoothing})',
)

## Differentiation

Use `simulate_heft` for gradients through the initial conditions and particle evolution. The saved-snapshot workflow reuses a fixed simulation and does not retain that upstream differentiation graph.

## Comparison with CSSTLaB

![Measured and emulated power spectra for all 15 pairs of HEFT operators](heft_validation/heft_csst_spectra.png)

The five auto-spectra and ten cross-spectra at $z=0$ compare our JAX HEFT calculation with [CSSTLaB](https://arxiv.org/abs/2506.04671). This figure comes from the standalone JAX HEFT implementation, in a separate run with box side $512\,h^{-1}\mathrm{Mpc}$, $512^3$ particles, a $768^3$ force mesh and a $256^3$ HEFT grid. It uses the CLASS linear cold-dark-matter-plus-baryon power spectrum at $z=1$, native pmwd growth, and the CSST Lagrangian window.

Over $0.15\le k<0.4\,h\,\mathrm{Mpc}^{-1}$, the median fractional differences of the five auto-spectra range from $-1.83\%$ to $-0.66\%$ in this realization. The $O_{d2}$–$O_L$ cross-spectrum changes sign within this interval at a lower $k$ in the simulation than in the emulator.

[Reproduce the figure and the calculation](heft_validation/README.md)